# ПР-4. Логистическая регрессия: предсказываем болезнь сердца

Датасет: `heart.csv` (Kaggle, Heart Failure Prediction). Файл должен лежать **в одной папке с этим ноутбуком**.

**Перед запуском** (один раз) установите библиотеки: создайте пустую ячейку, выполните
`%pip install pandas numpy matplotlib scikit-learn` и перезапустите ядро (Kernel → Restart).

Дальше запускайте блоки **по порядку сверху вниз** (Shift + Enter). Простое объяснение каждого блока — в файле `ПОЯСНЕНИЕ.md`.

## Блок 1. Загрузка данных и первый взгляд
Подключаем библиотеки, читаем таблицу и смотрим, что в ней есть.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, precision_score,
                             recall_score, f1_score, accuracy_score)

df = pd.read_csv("heart.csv")

print("Первые строки таблицы:")
print(df.head())
print("\nРазмер таблицы (строки, столбцы):", df.shape)
print("\nПропуски по столбцам:")
print(df.isna().sum())
print("\nСколько здоровых (0) и больных (1):")
print(df["HeartDisease"].value_counts())

## Блок 2. Очистка данных
В `Cholesterol` и `RestingBP` нули — это ошибки (так не бывает). Заменяем их на медиану остальных значений.

In [ ]:
for col in ["Cholesterol", "RestingBP"]:
    print(f"Нулей в {col}:", (df[col] == 0).sum())
    median = df.loc[df[col] != 0, col].median()   # медиана без нулей
    df[col] = df[col].replace(0, median)

print("\nПосле очистки нулей нет:")
print((df[["Cholesterol", "RestingBP"]] == 0).sum())

## Блок 3. Кодирование данных
Модель понимает только числа, поэтому текстовые столбцы превращаем в столбцы из 0 и 1 (one-hot кодирование).

In [ ]:
y = df["HeartDisease"]                  # то, что предсказываем
X = df.drop("HeartDisease", axis=1)     # признаки

text_columns = ["Sex", "ChestPainType", "RestingECG",
                "ExerciseAngina", "ST_Slope"]
X = pd.get_dummies(X, columns=text_columns, drop_first=True)
X = X.astype(float)                     # все значения делаем числами

print("Признаки после кодирования:")
print(list(X.columns))
print("\nКак теперь выглядит таблица:")
print(X.head())

## Блок 4. Разделение на обучающую и тестовую выборки
80% данных — для обучения, 20% — для проверки (модель их не видит при обучении).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки: ", X_test.shape)
print("Доля больных в обучающей:", round(y_train.mean(), 3))
print("Доля больных в тестовой: ", round(y_test.mean(), 3))

## Блок 5. Масштабирование признаков
Приводим все признаки к одному масштабу (среднее 0, разброс 1). Масштаб считаем по обучающей выборке и применяем к обеим.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Форма обучающей матрицы:", X_train_sc.shape)
print("Среднее первых 3 признаков (должно быть около 0):",
      X_train_sc[:, :3].mean(axis=0).round(3))

## Блок 6. Обучение модели из библиотеки sklearn
Одна строка `fit` подбирает веса модели. Затем получаем вероятности и предсказания.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)               # классы 0 или 1
y_proba = model.predict_proba(X_test_sc)[:, 1]  # вероятности класса 1

print("Первые 5 вероятностей болезни:", y_proba[:5].round(3))
print("Первые 5 предсказаний:        ", y_pred[:5])
print("Первые 5 реальных ответов:    ", y_test.values[:5])

## Блок 7. Метрики качества (precision, recall, F1)
Матрица ошибок, метрики из sklearn и те же метрики, посчитанные вручную по формулам.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
print("Матрица ошибок:")
print(cm)
print(f"TN={TN}, FP={FP}, FN={FN}, TP={TP}")

print("\nМетрики sklearn:")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1       :", round(f1_score(y_test, y_pred), 3))

# Те же метрики вручную по формулам (должны совпасть)
precision_hand = TP / (TP + FP)
recall_hand = TP / (TP + FN)
f1_hand = 2 * precision_hand * recall_hand / (precision_hand + recall_hand)

print("\nМетрики, посчитанные вручную:")
print("Precision:", round(precision_hand, 3))
print("Recall   :", round(recall_hand, 3))
print("F1       :", round(f1_hand, 3))

## Блок 8. Метод максимального правдоподобия «вручную» (главный блок для защиты)
Пишем обучение сами, без sklearn: сигмоида → вероятности → лог-правдоподобие → градиент → шаг вверх. Повторяем 3000 раз.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_likelihood(y, p):
    # ln L = сумма( y*ln(p) + (1-y)*ln(1-p) )
    p = np.clip(p, 1e-15, 1 - 1e-15)   # чтобы не было log(0)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

# Добавляем столбец из единиц - он нужен для свободного члена w0
ones_train = np.ones((X_train_sc.shape[0], 1))
ones_test = np.ones((X_test_sc.shape[0], 1))
Xb_train = np.hstack([ones_train, X_train_sc])
Xb_test = np.hstack([ones_test, X_test_sc])

y_train_arr = y_train.values
n = len(y_train_arr)

w = np.zeros(Xb_train.shape[1])   # начальные веса - нули
learning_rate = 0.1               # скорость обучения (размер шага)
n_iterations = 3000               # число шагов
history = []                      # здесь храним значения ln L

for i in range(n_iterations):
    z = Xb_train @ w                           # линейная комбинация
    p = sigmoid(z)                             # вероятности
    history.append(log_likelihood(y_train_arr, p))
    gradient = Xb_train.T @ (y_train_arr - p)  # градиент ln L
    w = w + learning_rate * gradient / n       # шаг ВВЕРХ (максимизируем)

print("Лог-правдоподобие в начале:", round(history[0], 2))
print("Лог-правдоподобие в конце: ", round(history[-1], 2))

# Предсказание собственной моделью
p_test = sigmoid(Xb_test @ w)
y_pred_my = (p_test >= 0.5).astype(int)

print("\nМетрики собственной модели:")
print("Precision:", round(precision_score(y_test, y_pred_my), 3))
print("Recall   :", round(recall_score(y_test, y_pred_my), 3))
print("F1       :", round(f1_score(y_test, y_pred_my), 3))

## Блок 9. Сравнение весов: своя модель и sklearn
Если всё сделано верно, веса почти совпадают. Для честного сравнения берём sklearn с огромным `C` (почти без регуляризации).

In [ ]:
model_plain = LogisticRegression(C=1e10, max_iter=5000)  # огромное C = почти без регуляризации
model_plain.fit(X_train_sc, y_train)

weights = pd.DataFrame({
    "признак": ["w0 (свободный член)"] + list(X.columns),
    "своя модель": w,
    "sklearn": np.concatenate([model_plain.intercept_, model_plain.coef_[0]])
})
print("Сравнение весов:")
print(weights.round(3).to_string(index=False))

## Блок 10. Графики
График роста лог-правдоподобия (модель «учится») и матрица ошибок.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("Номер итерации")
plt.ylabel("Лог-правдоподобие ln L")
plt.title("Рост лог-правдоподобия при обучении")
plt.grid(True)
plt.savefig("loglikelihood.png", dpi=150)
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Матрица ошибок")
plt.xlabel("Предсказано")
plt.ylabel("Реально")
plt.xticks([0, 1])
plt.yticks([0, 1])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()